# V5: Bounded Long-Horizon Agent Runtime

V5 studies the runtime semantics that become necessary when an agent must operate over
many steps instead of completing a task in one short loop. The goal is educational rather
than production deployment: every state transition is explicit, implemented in pure
Python, and exercised by a controlled end-to-end scenario.

The scenario intentionally follows this trajectory:

```text
transient tool failure
-> bounded retry
-> evidence-gap observation
-> PlanPatch replanning
-> human approval interrupt
-> serialized checkpoint
-> simulated process restart
-> resume with the same run ID
-> idempotent effects
-> report generation
-> trajectory evaluation
```

This notebook does not implement cross-run long-term memory. That is deliberately deferred
to V6 so V5 can isolate short-term working memory, resumability, and control flow.

## 0. Source Map and Design Rationale

V5 is based on recurring patterns in current agent engineering material:

- [OpenAI compaction](https://developers.openai.com/api/docs/guides/compaction): reduce
  growing context while preserving the state needed for later turns.
- [OpenAI agent evals](https://developers.openai.com/api/docs/guides/agent-evals): inspect
  and grade complete traces, including model calls, tools, guardrails, and handoffs.
- [Anthropic: Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents):
  compose routing, parallelization, orchestrator-worker, evaluator, and bounded agent loops.
- [Anthropic: Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents):
  preserve progress with structured handoff artifacts across long tasks.
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence):
  checkpoint state for recovery, replay, and human-in-the-loop execution.
- [AWS idempotency and retries](https://docs.aws.amazon.com/durable-execution/patterns/best-practices/idempotency/):
  retries and replay can execute an effect more than once, so effects require stable IDs.

We implement the ideas directly instead of importing an orchestration framework. The
distinction matters: a checkpoint is a recoverable state snapshot, working memory supports
the current task, a trace explains the trajectory, and an effect ledger prevents duplicate
external actions after retry or resume.

## 1. Setup

In [1]:
import hashlib
import json
import os
import platform
import sys
from collections import Counter
from datetime import UTC, datetime
from enum import StrEnum
from pathlib import Path
from typing import Any, Callable, Literal, Self
from uuid import uuid4

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field, model_validator

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path)

EXPECTED_PYTHON_VERSION = (3, 12)
assert sys.version_info[:2] == EXPECTED_PYTHON_VERSION

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5-nano")
USE_LIVE_MODEL = os.getenv("V5_USE_LIVE_MODEL", "0") == "1"

print(f"[SETUP] Python: {platform.python_version()}")
print(f"[SETUP] Project directory: {Path.cwd()}")
print(f"[SETUP] Model: {MODEL_NAME}")
print(f"[SETUP] Live Responses writer: {USE_LIVE_MODEL}")

[SETUP] Python: 3.12.12
[SETUP] Project directory: /Users/daboluo/MyWorkSpace/GitHub/agentic-deepresearch/notebooks
[SETUP] Model: gpt-5-nano
[SETUP] Live Responses writer: True


## 2. Runtime Contracts

A long-horizon harness needs explicit state. The model may propose actions, but deterministic
Python owns task status, budgets, plan versions, approval state, effect identity, and
termination.

In [2]:
class RunStatus(StrEnum):
    CREATED = "created"
    RUNNING = "running"
    REPLANNING = "replanning"
    WAITING_FOR_HUMAN = "waiting_for_human"
    COMPLETED = "completed"
    FAILED = "failed"
    BUDGET_EXHAUSTED = "budget_exhausted"
    CANCELLED = "cancelled"


class TaskStatus(StrEnum):
    PENDING = "pending"
    READY = "ready"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"
    BLOCKED = "blocked"
    SKIPPED = "skipped"


TaskKind = Literal["research", "write"]
EffectStatus = Literal["pending", "completed", "failed"]
HumanAction = Literal["approve", "edit", "reject", "respond"]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)


class SourceNote(StrictModel):
    id: str = Field(pattern=r"^src_[a-z0-9_]+$")
    title: str = Field(min_length=1)
    url: str = Field(min_length=1)
    source_type: Literal["academic", "industry", "documentation"]
    note: str = Field(min_length=1)


class Observation(StrictModel):
    task_id: str = Field(min_length=1)
    kind: Literal["tool_error", "research_result", "writer_result", "runtime"]
    summary: str = Field(min_length=1)


class TaskNode(StrictModel):
    id: str = Field(pattern=r"^[a-z][a-z0-9_]*$")
    kind: TaskKind
    description: str = Field(min_length=1)
    depends_on: list[str] = Field(default_factory=list)
    status: TaskStatus = TaskStatus.PENDING
    attempts: int = Field(default=0, ge=0)
    requires_approval: bool = False
    approved: bool = False
    artifact: str | None = None


class PlanVersion(StrictModel):
    version: int = Field(ge=1)
    reason: str = Field(min_length=1)
    tasks: list[TaskNode] = Field(min_length=1)


class PlanPatch(StrictModel):
    base_version: int = Field(ge=1)
    reason: str = Field(min_length=1)
    skip_task_ids: list[str] = Field(default_factory=list)
    add_tasks: list[TaskNode] = Field(default_factory=list)


class ExecutionBudget(StrictModel):
    max_steps: int = Field(default=8, ge=1)
    max_replans: int = Field(default=2, ge=0)
    max_tool_calls: int = Field(default=8, ge=1)
    max_failures: int = Field(default=3, ge=0)
    max_recent_observations: int = Field(default=1, ge=1)
    steps_used: int = Field(default=0, ge=0)
    replans_used: int = Field(default=0, ge=0)
    tool_calls_used: int = Field(default=0, ge=0)
    failures_used: int = Field(default=0, ge=0)


class WorkingMemory(StrictModel):
    goal: str = Field(min_length=1)
    sources: dict[str, SourceNote] = Field(default_factory=dict)
    observations: list[Observation] = Field(default_factory=list)
    open_questions: list[str] = Field(default_factory=list)
    decisions: list[str] = Field(default_factory=list)
    handoff_summary: str = ""
    compaction_count: int = Field(default=0, ge=0)


class EffectRecord(StrictModel):
    effect_id: str = Field(pattern=r"^eff_[0-9a-f]{12}$")
    task_id: str = Field(min_length=1)
    tool_name: str = Field(min_length=1)
    arguments_hash: str = Field(pattern=r"^[0-9a-f]{12}$")
    status: EffectStatus
    attempts: int = Field(default=0, ge=0)
    result: Any | None = None
    error: str | None = None


class HumanRequest(StrictModel):
    request_id: str = Field(pattern=r"^human_[0-9a-f]{12}$")
    task_id: str = Field(min_length=1)
    reason: str = Field(min_length=1)
    options: list[HumanAction] = Field(min_length=1)


class HumanDecision(StrictModel):
    request_id: str = Field(pattern=r"^human_[0-9a-f]{12}$")
    action: HumanAction
    content: str | None = None


class TraceEvent(StrictModel):
    sequence: int = Field(ge=1)
    event_type: str = Field(min_length=1)
    detail: str = Field(min_length=1)
    task_id: str | None = None
    timestamp: str


class LongHorizonRunState(StrictModel):
    run_id: str = Field(pattern=r"^run_[0-9a-f]{12}$")
    goal: str = Field(min_length=1)
    status: RunStatus = RunStatus.CREATED
    plan_versions: list[PlanVersion] = Field(min_length=1)
    active_plan_version: int = Field(ge=1)
    memory: WorkingMemory
    budget: ExecutionBudget
    effects: dict[str, EffectRecord] = Field(default_factory=dict)
    trace: list[TraceEvent] = Field(default_factory=list)
    pending_human_request: HumanRequest | None = None
    final_report: str | None = None
    last_checkpoint_id: str | None = None

    @model_validator(mode="after")
    def validate_active_version(self) -> Self:
        versions = [plan.version for plan in self.plan_versions]
        if versions != list(range(1, len(versions) + 1)):
            raise ValueError("Plan versions must be consecutive and one-based.")
        if self.active_plan_version != versions[-1]:
            raise ValueError("The active plan must be the newest plan version.")
        return self


class Checkpoint(StrictModel):
    checkpoint_id: str = Field(pattern=r"^cp_[0-9]{3}$")
    run_id: str = Field(pattern=r"^run_[0-9a-f]{12}$")
    reason: str = Field(min_length=1)
    state_json: str = Field(min_length=1)
    created_at: str

### 2.1 Plan invariants

Replanning is safe only when the harness validates the new graph. Completed work is copied
into the next plan version, dependencies must exist, and cycles are rejected.

In [3]:
TERMINAL_TASK_STATUSES = {
    TaskStatus.COMPLETED,
    TaskStatus.FAILED,
    TaskStatus.BLOCKED,
    TaskStatus.SKIPPED,
}


def validate_plan(plan: PlanVersion) -> None:
    task_ids = [task.id for task in plan.tasks]
    if len(task_ids) != len(set(task_ids)):
        raise ValueError("Task IDs must be unique within a plan version.")

    known_ids = set(task_ids)
    for task in plan.tasks:
        missing = set(task.depends_on) - known_ids
        if missing:
            raise ValueError(f"Task {task.id!r} has missing dependencies: {sorted(missing)}")
        if task.id in task.depends_on:
            raise ValueError(f"Task {task.id!r} cannot depend on itself.")

    visiting: set[str] = set()
    visited: set[str] = set()
    dependencies = {task.id: task.depends_on for task in plan.tasks}

    def visit(task_id: str) -> None:
        if task_id in visiting:
            raise ValueError("The task graph must be acyclic.")
        if task_id in visited:
            return
        visiting.add(task_id)
        for dependency_id in dependencies[task_id]:
            visit(dependency_id)
        visiting.remove(task_id)
        visited.add(task_id)

    for task_id in task_ids:
        visit(task_id)


def active_plan(state: LongHorizonRunState) -> PlanVersion:
    return state.plan_versions[-1]


def find_task(state: LongHorizonRunState, task_id: str) -> TaskNode:
    for task in active_plan(state).tasks:
        if task.id == task_id:
            return task
    raise KeyError(f"Unknown task ID: {task_id}")

## 3. Trace, Checkpoint, and Effect Primitives

In [4]:
def utc_now() -> str:
    return datetime.now(UTC).isoformat()


def append_event(
    state: LongHorizonRunState,
    event_type: str,
    detail: str,
    task_id: str | None = None,
) -> None:
    state.trace.append(
        TraceEvent(
            sequence=len(state.trace) + 1,
            event_type=event_type,
            detail=detail,
            task_id=task_id,
            timestamp=utc_now(),
        )
    )


class InMemoryCheckpointer:
    # Serializable storage used to demonstrate process restart semantics.

    def __init__(self) -> None:
        self._items: dict[str, Checkpoint] = {}

    def save(self, state: LongHorizonRunState, reason: str) -> str:
        checkpoint_id = f"cp_{len(self._items) + 1:03d}"
        state.last_checkpoint_id = checkpoint_id
        append_event(state, "checkpoint_saved", reason)
        checkpoint = Checkpoint(
            checkpoint_id=checkpoint_id,
            run_id=state.run_id,
            reason=reason,
            state_json=state.model_dump_json(),
            created_at=utc_now(),
        )
        self._items[checkpoint_id] = checkpoint
        return checkpoint_id

    def load(self, checkpoint_id: str) -> LongHorizonRunState:
        checkpoint = self._items[checkpoint_id]
        return LongHorizonRunState.model_validate_json(checkpoint.state_json)

    def export_json(self) -> str:
        payload = {
            checkpoint_id: checkpoint.model_dump(mode="json")
            for checkpoint_id, checkpoint in self._items.items()
        }
        return json.dumps(payload, sort_keys=True)

    @classmethod
    def from_json(cls, payload: str) -> Self:
        restored = cls()
        for checkpoint_id, item in json.loads(payload).items():
            restored._items[checkpoint_id] = Checkpoint.model_validate(item)
        return restored

    def __len__(self) -> int:
        return len(self._items)


def stable_effect_id(task_id: str, tool_name: str, arguments: dict[str, Any]) -> tuple[str, str]:
    normalized = json.dumps(arguments, sort_keys=True, separators=(",", ":"))
    arguments_hash = hashlib.sha256(normalized.encode()).hexdigest()[:12]
    raw = f"{task_id}|{tool_name}|{arguments_hash}"
    effect_id = f"eff_{hashlib.sha256(raw.encode()).hexdigest()[:12]}"
    return effect_id, arguments_hash

### 3.1 Idempotent effect execution

The effect ledger stores one record per logical tool action. A retry reuses the same effect
ID, and a completed effect returns its previous result instead of calling the tool again.

In [5]:
def execute_effect(
    state: LongHorizonRunState,
    *,
    task_id: str,
    tool_name: str,
    arguments: dict[str, Any],
    operation: Callable[..., Any],
    max_attempts: int = 2,
) -> Any:
    effect_id, arguments_hash = stable_effect_id(task_id, tool_name, arguments)
    existing = state.effects.get(effect_id)

    if existing is not None and existing.status == "completed":
        append_event(
            state,
            "effect_reused",
            f"Reused completed effect {effect_id}.",
            task_id,
        )
        return existing.result

    record = existing or EffectRecord(
        effect_id=effect_id,
        task_id=task_id,
        tool_name=tool_name,
        arguments_hash=arguments_hash,
        status="pending",
    )
    state.effects[effect_id] = record

    while record.attempts < max_attempts:
        if state.budget.tool_calls_used >= state.budget.max_tool_calls:
            raise RuntimeError("Tool-call budget exhausted.")

        record.attempts += 1
        state.budget.tool_calls_used += 1
        append_event(
            state,
            "effect_started",
            f"Calling {tool_name}; attempt {record.attempts}.",
            task_id,
        )

        try:
            result = operation(**arguments)
        except Exception as exc:
            record.status = "failed"
            record.error = f"{type(exc).__name__}: {exc}"
            state.budget.failures_used += 1
            state.memory.observations.append(
                Observation(
                    task_id=task_id,
                    kind="tool_error",
                    summary=f"{tool_name} attempt {record.attempts} failed transiently.",
                )
            )
            append_event(state, "effect_failed", record.error, task_id)

            if (
                record.attempts >= max_attempts
                or state.budget.failures_used > state.budget.max_failures
            ):
                raise

            record.status = "pending"
            append_event(
                state,
                "effect_retry",
                f"Retrying {effect_id} within the bounded retry policy.",
                task_id,
            )
            continue

        record.status = "completed"
        record.result = result
        record.error = None
        append_event(state, "effect_completed", f"Completed {effect_id}.", task_id)
        return result

    raise RuntimeError(f"Effect {effect_id} exhausted its attempts.")

## 4. Working Memory and Context Compaction

In [6]:
def compact_working_memory(state: LongHorizonRunState) -> None:
    limit = state.budget.max_recent_observations
    observations = state.memory.observations
    if len(observations) <= limit:
        return

    old_items = observations[:-limit]
    state.memory.observations = observations[-limit:]
    compacted_lines = [f"- {item.task_id}: {item.summary}" for item in old_items]
    previous = state.memory.handoff_summary.strip()
    new_summary = "\n".join(compacted_lines)
    state.memory.handoff_summary = "\n".join(
        part for part in [previous, new_summary] if part
    )
    state.memory.compaction_count += 1
    append_event(
        state,
        "context_compacted",
        f"Compacted {len(old_items)} observations; retained {limit} recent item(s).",
    )


def build_writer_context(state: LongHorizonRunState) -> dict[str, Any]:
    return {
        "goal": state.goal,
        "handoff_summary": state.memory.handoff_summary,
        "recent_observations": [
            observation.model_dump(mode="json")
            for observation in state.memory.observations
        ],
        "open_questions": state.memory.open_questions,
        "sources": [
            source.model_dump(mode="json")
            for source in state.memory.sources.values()
        ],
    }

## 5. Initial Planning and PlanPatch Replanning

In [7]:
def build_initial_plan() -> PlanVersion:
    plan = PlanVersion(
        version=1,
        reason="Initial bounded research plan.",
        tasks=[
            TaskNode(
                id="research_web",
                kind="research",
                description="Collect current industry and documentation evidence.",
            ),
            TaskNode(
                id="write_initial",
                kind="write",
                description="Write the final source-backed Markdown report.",
                depends_on=["research_web"],
            ),
        ],
    )
    validate_plan(plan)
    return plan


def build_evidence_gap_patch(state: LongHorizonRunState) -> PlanPatch:
    return PlanPatch(
        base_version=state.active_plan_version,
        reason=(
            "The initial search found useful engineering guidance but no academic "
            "long-horizon source. Add a focused evidence lane before writing."
        ),
        skip_task_ids=["write_initial"],
        add_tasks=[
            TaskNode(
                id="research_academic",
                kind="research",
                description="Collect academic and durable-execution evidence.",
                depends_on=["research_web"],
                requires_approval=True,
            ),
            TaskNode(
                id="write_final",
                kind="write",
                description="Synthesize the final source-backed Markdown report.",
                depends_on=["research_web", "research_academic"],
            ),
        ],
    )


def apply_plan_patch(state: LongHorizonRunState, patch: PlanPatch) -> PlanVersion:
    if patch.base_version != state.active_plan_version:
        raise ValueError("PlanPatch must target the active plan version.")
    if state.budget.replans_used >= state.budget.max_replans:
        raise RuntimeError("Replanning budget exhausted.")

    previous = active_plan(state)
    next_tasks = [task.model_copy(deep=True) for task in previous.tasks]
    task_by_id = {task.id: task for task in next_tasks}

    for task_id in patch.skip_task_ids:
        task = task_by_id[task_id]
        if task.status == TaskStatus.COMPLETED:
            raise ValueError("Completed tasks are immutable across replanning.")
        task.status = TaskStatus.SKIPPED

    existing_ids = set(task_by_id)
    for task in patch.add_tasks:
        if task.id in existing_ids:
            raise ValueError(f"PlanPatch cannot add duplicate task {task.id!r}.")
        next_tasks.append(task.model_copy(deep=True))
        existing_ids.add(task.id)

    next_plan = PlanVersion(
        version=previous.version + 1,
        reason=patch.reason,
        tasks=next_tasks,
    )
    validate_plan(next_plan)

    state.status = RunStatus.REPLANNING
    state.plan_versions.append(next_plan)
    state.active_plan_version = next_plan.version
    state.budget.replans_used += 1
    state.memory.decisions.append(patch.reason)
    state.status = RunStatus.RUNNING
    append_event(state, "replan_applied", patch.reason)
    return next_plan

## 6. Worker Adapters

The runtime depends on small worker interfaces. The controlled research tool is deterministic
so failure and recovery behavior remain reproducible. The writer can use either a deterministic
implementation or one live OpenAI Responses API call.

In [8]:
class ControlledResearchTool:
    def __init__(self) -> None:
        self.call_counts: Counter[str] = Counter()

    def search(self, lane: str) -> list[dict[str, str]]:
        self.call_counts[lane] += 1

        if lane == "industry" and self.call_counts[lane] == 1:
            raise ConnectionError("Injected transient search failure.")

        if lane == "industry":
            return [
                SourceNote(
                    id="src_anthropic_agents",
                    title="Building Effective Agents",
                    url="https://www.anthropic.com/engineering/building-effective-agents",
                    source_type="industry",
                    note=(
                        "Use simple composable workflows, environmental feedback, "
                        "stopping conditions, and human checkpoints."
                    ),
                ).model_dump(mode="json"),
                SourceNote(
                    id="src_openai_compaction",
                    title="OpenAI Compaction Guide",
                    url="https://developers.openai.com/api/docs/guides/compaction",
                    source_type="documentation",
                    note=(
                        "Long-running interactions require context reduction that preserves "
                        "state needed by subsequent turns."
                    ),
                ).model_dump(mode="json"),
            ]

        if lane == "academic":
            return [
                SourceNote(
                    id="src_horizon_gap",
                    title="The Horizon Gap",
                    url="https://arxiv.org/abs/2608.06663",
                    source_type="academic",
                    note=(
                        "Long-horizon reliability requires coordinated planning, memory, "
                        "execution, evaluation, and safety mechanisms."
                    ),
                ).model_dump(mode="json"),
                SourceNote(
                    id="src_langgraph_persistence",
                    title="LangGraph Persistence",
                    url="https://docs.langchain.com/oss/python/langgraph/persistence",
                    source_type="documentation",
                    note=(
                        "Checkpointed state supports fault recovery, replay, memory, and "
                        "human-in-the-loop workflows."
                    ),
                ).model_dump(mode="json"),
                SourceNote(
                    id="src_aws_idempotency",
                    title="AWS Idempotency and Retries",
                    url=(
                        "https://docs.aws.amazon.com/durable-execution/patterns/"
                        "best-practices/idempotency/"
                    ),
                    source_type="documentation",
                    note=(
                        "Replay and retry may execute an effect more than once, so external "
                        "effects need idempotency keys or equivalent deduplication."
                    ),
                ).model_dump(mode="json"),
            ]

        raise ValueError(f"Unknown research lane: {lane}")


def deterministic_report(context: dict[str, Any]) -> str:
    lines = [
        "# Bounded Long-Horizon Agent Design",
        "",
        "## Executive Summary",
        "",
        (
            "Reliable long-horizon agents need more than a larger iteration limit. They "
            "need explicit state, observation-triggered replanning, bounded autonomy, "
            "recoverable checkpoints, safe effect semantics, and trajectory evaluation."
        ),
        "",
        "## Findings",
        "",
    ]
    for source in context["sources"]:
        lines.append(f"- {source['note']} [{source['id']}]")

    lines.extend(["", "## Runtime Design", ""])
    lines.append(
        "The supervisor owns deterministic budgets and state transitions. The model may "
        "recommend actions, but Python validates plan patches, approvals, dependencies, "
        "and terminal conditions. Working memory retains structured evidence while older "
        "observations are compacted into a bounded handoff summary."
    )
    lines.extend(["", "## References", ""])
    for source in context["sources"]:
        lines.append(f"- [{source['id']}] {source['title']}: {source['url']}")
    return "\n".join(lines)


def responses_report(context: dict[str, Any], model: str = MODEL_NAME) -> str:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is required for the live writer.")

    client = OpenAI()
    response = client.responses.create(
        model=model,
        instructions=(
            "Write a concise Markdown research report. Use every provided source at least "
            "once with its exact source ID in square brackets. Include Executive Summary, "
            "Findings, Runtime Design, Limitations, and References. Output only the report."
        ),
        input=[
            {
                "role": "user",
                "content": json.dumps(context, indent=2),
            }
        ],
    )
    report = response.output_text.strip()
    if not report:
        raise RuntimeError("The live writer returned an empty report.")
    return report

## 7. Bounded Long-Horizon Runtime

In [9]:
class LongHorizonRuntime:
    def __init__(
        self,
        state: LongHorizonRunState,
        checkpointer: InMemoryCheckpointer,
        research_tool: ControlledResearchTool,
        report_writer: Callable[[dict[str, Any]], str],
    ) -> None:
        self.state = state
        self.checkpointer = checkpointer
        self.research_tool = research_tool
        self.report_writer = report_writer

    @classmethod
    def new(
        cls,
        *,
        goal: str,
        checkpointer: InMemoryCheckpointer,
        research_tool: ControlledResearchTool,
        report_writer: Callable[[dict[str, Any]], str],
    ) -> Self:
        plan = build_initial_plan()
        state = LongHorizonRunState(
            run_id=f"run_{uuid4().hex[:12]}",
            goal=goal,
            plan_versions=[plan],
            active_plan_version=1,
            memory=WorkingMemory(goal=goal),
            budget=ExecutionBudget(),
        )
        return cls(state, checkpointer, research_tool, report_writer)

    @classmethod
    def from_checkpoint(
        cls,
        *,
        checkpoint_id: str,
        checkpointer: InMemoryCheckpointer,
        research_tool: ControlledResearchTool,
        report_writer: Callable[[dict[str, Any]], str],
    ) -> Self:
        state = checkpointer.load(checkpoint_id)
        append_event(
            state,
            "run_resumed",
            f"Restored state from {checkpoint_id}.",
        )
        return cls(state, checkpointer, research_tool, report_writer)

    def _refresh_task_states(self) -> None:
        task_by_id = {task.id: task for task in active_plan(self.state).tasks}
        for task in active_plan(self.state).tasks:
            if task.status in TERMINAL_TASK_STATUSES or task.status == TaskStatus.RUNNING:
                continue
            dependency_states = [task_by_id[item].status for item in task.depends_on]
            if any(
                status in {TaskStatus.FAILED, TaskStatus.BLOCKED, TaskStatus.SKIPPED}
                for status in dependency_states
            ):
                task.status = TaskStatus.BLOCKED
            elif all(status == TaskStatus.COMPLETED for status in dependency_states):
                task.status = TaskStatus.READY
            else:
                task.status = TaskStatus.PENDING

    def _needs_replan(self) -> bool:
        has_gap = any(
            "academic" in question.lower()
            for question in self.state.memory.open_questions
        )
        has_academic_task = any(
            task.id == "research_academic"
            for task in active_plan(self.state).tasks
        )
        return has_gap and not has_academic_task

    def _request_human_approval(self, task: TaskNode) -> str:
        request = HumanRequest(
            request_id=f"human_{uuid4().hex[:12]}",
            task_id=task.id,
            reason=(
                "The evidence gap requires an additional research lane. Approval is "
                "required because it expands the original execution budget."
            ),
            options=["approve", "edit", "reject", "respond"],
        )
        self.state.pending_human_request = request
        self.state.status = RunStatus.WAITING_FOR_HUMAN
        append_event(self.state, "human_requested", request.reason, task.id)
        return self.checkpointer.save(self.state, "human_interrupt")

    def apply_human_decision(self, decision: HumanDecision) -> None:
        request = self.state.pending_human_request
        if request is None:
            raise RuntimeError("No human decision is currently pending.")
        if decision.request_id != request.request_id:
            raise ValueError("Human decision does not match the pending request.")

        task = find_task(self.state, request.task_id)
        if decision.action == "reject":
            task.status = TaskStatus.SKIPPED
            self.state.status = RunStatus.CANCELLED
        else:
            if decision.action == "edit":
                if not decision.content:
                    raise ValueError("An edit decision requires replacement task content.")
                task.description = decision.content
            if decision.action == "respond" and decision.content:
                self.state.memory.decisions.append(decision.content)
            task.approved = True
            self.state.status = RunStatus.RUNNING

        self.state.memory.decisions.append(
            f"Human {decision.action} for task {task.id}."
        )
        self.state.pending_human_request = None
        append_event(
            self.state,
            "human_decision",
            f"Human selected {decision.action}.",
            task.id,
        )
        self.checkpointer.save(self.state, "human_decision")

    def _execute_research(self, task: TaskNode) -> None:
        lane = "industry" if task.id == "research_web" else "academic"
        raw_sources = execute_effect(
            self.state,
            task_id=task.id,
            tool_name="controlled_research_search",
            arguments={"lane": lane},
            operation=self.research_tool.search,
        )
        sources = [SourceNote.model_validate(item) for item in raw_sources]
        for source in sources:
            self.state.memory.sources[source.id] = source

        if lane == "industry":
            gap = "Academic evidence is still required for long-horizon claims."
            if gap not in self.state.memory.open_questions:
                self.state.memory.open_questions.append(gap)
        else:
            self.state.memory.open_questions = [
                item
                for item in self.state.memory.open_questions
                if "academic" not in item.lower()
            ]

        task.artifact = ", ".join(source.id for source in sources)
        self.state.memory.observations.append(
            Observation(
                task_id=task.id,
                kind="research_result",
                summary=f"Collected {len(sources)} source records for the {lane} lane.",
            )
        )

    def _execute_write(self, task: TaskNode) -> None:
        context = build_writer_context(self.state)
        report = self.report_writer(context)
        task.artifact = report
        self.state.final_report = report
        self.state.memory.observations.append(
            Observation(
                task_id=task.id,
                kind="writer_result",
                summary=f"Generated a {len(report)}-character Markdown report.",
            )
        )

    def _execute_task(self, task: TaskNode) -> None:
        if self.state.budget.steps_used >= self.state.budget.max_steps:
            self.state.status = RunStatus.BUDGET_EXHAUSTED
            append_event(self.state, "budget_exhausted", "Step budget exhausted.")
            return

        task.status = TaskStatus.RUNNING
        task.attempts += 1
        self.state.budget.steps_used += 1
        append_event(self.state, "task_started", task.description, task.id)

        try:
            if task.kind == "research":
                self._execute_research(task)
            elif task.kind == "write":
                self._execute_write(task)
            else:
                raise ValueError(f"Unsupported task kind: {task.kind}")
        except Exception as exc:
            task.status = TaskStatus.FAILED
            self.state.status = RunStatus.FAILED
            append_event(
                self.state,
                "task_failed",
                f"{type(exc).__name__}: {exc}",
                task.id,
            )
            self.checkpointer.save(self.state, f"task_failed:{task.id}")
            raise

        task.status = TaskStatus.COMPLETED
        append_event(self.state, "task_completed", task.artifact or "Completed.", task.id)
        compact_working_memory(self.state)
        self.checkpointer.save(self.state, f"task_completed:{task.id}")

    def run_until_pause_or_complete(self) -> LongHorizonRunState:
        if self.state.status == RunStatus.CREATED:
            self.state.status = RunStatus.RUNNING
            append_event(self.state, "run_started", self.state.goal)

        while self.state.status == RunStatus.RUNNING:
            if self._needs_replan():
                patch = build_evidence_gap_patch(self.state)
                apply_plan_patch(self.state, patch)
                self.checkpointer.save(self.state, "replan_applied")

            self._refresh_task_states()
            ready_tasks = [
                task
                for task in active_plan(self.state).tasks
                if task.status == TaskStatus.READY
            ]

            if not ready_tasks:
                active_tasks = active_plan(self.state).tasks
                if self.state.final_report and all(
                    task.status in {TaskStatus.COMPLETED, TaskStatus.SKIPPED}
                    for task in active_tasks
                ):
                    self.state.status = RunStatus.COMPLETED
                    append_event(self.state, "run_completed", "All active tasks completed.")
                    self.checkpointer.save(self.state, "run_completed")
                else:
                    self.state.status = RunStatus.FAILED
                    append_event(
                        self.state,
                        "run_failed",
                        "No runnable task remained before the goal was complete.",
                    )
                    self.checkpointer.save(self.state, "run_failed")
                break

            task = ready_tasks[0]
            if task.requires_approval and not task.approved:
                self._request_human_approval(task)
                break

            self._execute_task(task)

        return self.state

## 8. End-to-End Evaluation Run

In [10]:
GOAL = (
    "Explain the engineering mechanisms required for a reliable bounded long-horizon "
    "research agent and produce a source-backed Markdown report."
)

checkpointer = InMemoryCheckpointer()
controlled_tool = ControlledResearchTool()
writer = responses_report if USE_LIVE_MODEL else deterministic_report

runtime = LongHorizonRuntime.new(
    goal=GOAL,
    checkpointer=checkpointer,
    research_tool=controlled_tool,
    report_writer=writer,
)
original_run_id = runtime.state.run_id

paused_state = runtime.run_until_pause_or_complete()

print(f"[V5] Phase 1 status: {paused_state.status.value}")
print(f"[V5] Plan versions: {len(paused_state.plan_versions)}")
print(f"[V5] Replans used: {paused_state.budget.replans_used}")
print(f"[V5] Tool calls used: {paused_state.budget.tool_calls_used}")
print(f"[V5] Pending request: {paused_state.pending_human_request.request_id}")
print(f"[V5] Latest checkpoint: {paused_state.last_checkpoint_id}")

assert paused_state.status == RunStatus.WAITING_FOR_HUMAN
assert paused_state.budget.replans_used == 1
assert paused_state.pending_human_request is not None
assert find_task(paused_state, "research_web").status == TaskStatus.COMPLETED

[V5] Phase 1 status: waiting_for_human
[V5] Plan versions: 2
[V5] Replans used: 1
[V5] Tool calls used: 2
[V5] Pending request: human_d0d241cbf107
[V5] Latest checkpoint: cp_003


### 8.1 Simulated process restart and human approval

Exporting and importing the checkpointer simulates losing the original runtime object. The
new runtime receives only serialized checkpoint data and the same worker adapters.

In [11]:
checkpoint_bundle = checkpointer.export_json()
latest_checkpoint_id = paused_state.last_checkpoint_id
pending_request_id = paused_state.pending_human_request.request_id

restarted_checkpointer = InMemoryCheckpointer.from_json(checkpoint_bundle)
restarted_runtime = LongHorizonRuntime.from_checkpoint(
    checkpoint_id=latest_checkpoint_id,
    checkpointer=restarted_checkpointer,
    research_tool=controlled_tool,
    report_writer=writer,
)

restarted_runtime.apply_human_decision(
    HumanDecision(
        request_id=pending_request_id,
        action="approve",
        content="Approve one additional focused academic evidence lane.",
    )
)
final_state = restarted_runtime.run_until_pause_or_complete()

print(f"[V5] Final status: {final_state.status.value}")
print(f"[V5] Same run ID after restart: {final_state.run_id == original_run_id}")
print(f"[V5] Active plan version: {final_state.active_plan_version}")
print(f"[V5] Completed steps: {final_state.budget.steps_used}")
print(f"[V5] Tool calls: {final_state.budget.tool_calls_used}")
print(f"[V5] Compactions: {final_state.memory.compaction_count}")
print(f"[V5] Checkpoints: {len(restarted_checkpointer)}")
print(f"[V5] Report characters: {len(final_state.final_report or '')}")

[V5] Final status: completed
[V5] Same run ID after restart: True
[V5] Active plan version: 2
[V5] Completed steps: 3
[V5] Tool calls: 3
[V5] Compactions: 3
[V5] Checkpoints: 7
[V5] Report characters: 5167


### 8.2 Trajectory graders

In [12]:
class V5EvaluationResult(StrictModel):
    passed: bool
    checks: dict[str, bool]
    metrics: dict[str, int | float | str]


def evaluate_v5_run(state: LongHorizonRunState) -> V5EvaluationResult:
    event_counts = Counter(event.event_type for event in state.trace)
    task_completions = Counter(
        event.task_id
        for event in state.trace
        if event.event_type == "task_completed" and event.task_id is not None
    )
    completed_effects = [
        effect for effect in state.effects.values() if effect.status == "completed"
    ]
    all_sources_cited = all(
        source_id in (state.final_report or "")
        for source_id in state.memory.sources
    )
    completed_work_preserved = (
        find_task(state, "research_web").status == TaskStatus.COMPLETED
        and task_completions["research_web"] == 1
    )
    required_events = {
        "effect_failed",
        "effect_retry",
        "effect_completed",
        "replan_applied",
        "human_requested",
        "run_resumed",
        "human_decision",
        "context_compacted",
        "run_completed",
    }

    checks = {
        "run_completed": state.status == RunStatus.COMPLETED,
        "same_run_resumed": state.run_id == original_run_id,
        "bounded_replanning": state.budget.replans_used == 1,
        "completed_work_preserved": completed_work_preserved,
        "effects_are_unique": len(state.effects) == len({item.effect_id for item in completed_effects}),
        "retry_reused_effect_id": any(effect.attempts == 2 for effect in completed_effects),
        "human_decision_consumed_once": event_counts["human_decision"] == 1,
        "working_memory_compacted": state.memory.compaction_count >= 1,
        "bounded_recent_context": (
            len(state.memory.observations) <= state.budget.max_recent_observations
        ),
        "all_sources_cited": all_sources_cited,
        "required_trace_events": required_events <= set(event_counts),
        "budgets_respected": (
            state.budget.steps_used <= state.budget.max_steps
            and state.budget.replans_used <= state.budget.max_replans
            and state.budget.tool_calls_used <= state.budget.max_tool_calls
        ),
    }
    metrics: dict[str, int | float | str] = {
        "plan_versions": len(state.plan_versions),
        "trace_events": len(state.trace),
        "tool_calls": state.budget.tool_calls_used,
        "effects": len(state.effects),
        "checkpoints": len(restarted_checkpointer),
        "sources": len(state.memory.sources),
        "report_characters": len(state.final_report or ""),
        "writer_mode": "openai_responses" if USE_LIVE_MODEL else "deterministic",
    }
    return V5EvaluationResult(
        passed=all(checks.values()),
        checks=checks,
        metrics=metrics,
    )


evaluation = evaluate_v5_run(final_state)
print(evaluation.model_dump_json(indent=2))
assert evaluation.passed, evaluation.model_dump_json(indent=2)

{
  "passed": true,
  "checks": {
    "run_completed": true,
    "same_run_resumed": true,
    "bounded_replanning": true,
    "completed_work_preserved": true,
    "effects_are_unique": true,
    "retry_reused_effect_id": true,
    "human_decision_consumed_once": true,
    "working_memory_compacted": true,
    "bounded_recent_context": true,
    "all_sources_cited": true,
    "required_trace_events": true,
    "budgets_respected": true
  },
  "metrics": {
    "plan_versions": 2,
    "trace_events": 29,
    "tool_calls": 3,
    "effects": 2,
    "checkpoints": 7,
    "sources": 5,
    "report_characters": 5167,
    "writer_mode": "openai_responses"
  }
}


## 9. Embedded Final Report

In [13]:
from IPython.display import Markdown, display

display(Markdown(final_state.final_report or "No report was produced."))

# Executive Summary
To engineer a reliable bounded long-horizon research agent, adopt a design that uses simple composable workflows, environmental feedback, stopping conditions, and human checkpoints to bound and govern behavior [src_anthropic_agents]. Long-running interactions must employ context compaction that preserves state needed by subsequent turns, ensuring the agent can reason over extended horizons without unbounded memory growth [src_openai_compaction]. A memory and checkpointed state subsystem enables fault recovery, replay, persistent memory, and human-in-the-loop workflows to maintain reliability across runs [src_langgraph_persistence]. External effects should be idempotent or deduplicated to tolerate replay and retries without duplicating real-world impact [src_aws_idempotency]. Together, these elements support a reliable architecture organized around coordinated planning, memory, execution, evaluation, and safety mechanisms for long-horizon tasks [src_horizon_gap].

# Findings
- Architectural foundation for long-horizon reliability relies on coordinated planning, explicit memory, controlled execution, continuous evaluation, and safety mechanisms [src_horizon_gap].

- Simple composable workflows, environmental feedback, stopping conditions, and human checkpoints enable bounded, auditable behavior with risk-limiting gates [src_anthropic_agents].

- Context management via context compaction preserves essential state across turns, enabling continuation over long horizons without unbounded growth [src_openai_compaction].

- Checkpointed memory supports fault recovery, replay, persistent memory, and human-in-the-loop workflows, providing resilience to interruptions and model drift [src_langgraph_persistence].

- External actions and side effects must be treated with idempotency or robust deduplication to prevent duplicate effects during replay or retries [src_aws_idempotency].

# Runtime Design
- Bounded horizon controller: encodes stopping conditions, budget constraints (time, steps, or tokens), and environmental feedback to keep progress within the desired horizon while maintaining safety and traceability. This aligns with guidance on stopping conditions and bounded workflows [src_anthropic_agents].

- Context management and compaction: maintain a compact, relevant history that preserves information needed for future turns, enabling efficient reasoning over long horizons without exhausting resources [src_openai_compaction].

- Memory and persistence layer: implement a checkpointed state system that supports fault recovery, replay, memory, and human-in-the-loop workflows to sustain reliability across sessions [src_langgraph_persistence].

- Coordinated planning, execution, evaluation, and safety: enforce a loop where a plan is formed, executed, evaluated, and adjusted with safety checks and gates to prevent unsafe or undesired outcomes [src_horizon_gap].

- Idempotent external effects: attach idempotency keys or deduplication mechanisms to all externally observable actions to prevent repeated real-world effects in the presence of replay or retries [src_aws_idempotency].

- Human-in-the-loop checkpoints: insert human review points at critical junctures to validate plan viability, safety, and alignment with research goals, reducing the likelihood of unsafe autonomous action [src_anthropic_agents].

- Integration pattern: Planner → Executor → Evaluator with memory-backed state (checkpointed) and safety monitors, all backed by compaction to maintain a bounded knowledge base and enable replay without unrecoverable drift [src_horizon_gap], [src_langgraph_persistence], [src_openai_compaction].

# Limitations
- Complexity and integration overhead: The combination of planning, memory/persistence, compaction, and safety gates increases system complexity and operational overhead.

- Dependence on external services: Reliability and latency of external systems (e.g., persistence, safety checks) can become bottlenecks or single points of failure.

- Memory vs. fidelity trade-offs: While compaction reduces resource use, it risks discarding information that could become important for future decisions if not carefully managed [src_openai_compaction].

- Safety and evaluation rigor: Ensuring robust evaluation and safety across all possible long-horizon scenarios remains challenging; failures may occur despite structured gates.

- Idempotency guarantees across diverse effects: Designing universal idempotency keys for all external actions can be nontrivial, especially for complex or non-idempotent domains [src_aws_idempotency].

# References
- [src_anthropic_agents] Building Effective Agents — https://www.anthropic.com/engineering/building-effective-agents
- [src_openai_compaction] OpenAI Compaction Guide — https://developers.openai.com/api/docs/guides/compaction
- [src_horizon_gap] The Horizon Gap — https://arxiv.org/abs/2608.06663
- [src_langgraph_persistence] LangGraph Persistence — https://docs.langchain.com/oss/python/langgraph/persistence
- [src_aws_idempotency] AWS Idempotency and Retries — https://docs.aws.amazon.com/durable-execution/patterns/best-practices/idempotency/

## 10. V5 Completion Summary

V5 adds a bounded long-horizon runtime without hiding orchestration inside a framework:

- a versioned task graph and validated `PlanPatch` replanning;
- run-scoped working memory with explicit compaction and structured source retention;
- serialized checkpoints and reconstruction of a new runtime object;
- a first-class `WAITING_FOR_HUMAN` state and typed human decisions;
- an effect ledger with stable IDs and bounded retry semantics;
- deterministic budgets and termination owned by Python rather than the model;
- controlled failure injection and trajectory-level evaluation.

The end-to-end run proves that a completed task survives replanning and restart, a transient
failure retries the same logical effect, execution pauses before scope expansion, and the
final report is produced only after the human decision is applied.

Deferred to V6: cross-run episodic, semantic, and procedural long-term memory; retrieval and
write policies; consolidation; conflicts; provenance; and forgetting.